In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from pathlib import Path
import pickle
import logging
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class ContextualEmbeddingGenerator:

    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        logger.info(f"Loading model: {model_name}")
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def mean_pooling(self, token_embeddings, attention_mask):
        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)

    def get_word_embedding(self, sentence, target_word):
        encoded = self.tokenizer(sentence, return_tensors="pt", truncation=True, max_length=512).to(self.device)

        with torch.no_grad():
            outputs = self.model(**encoded)

        token_embeddings = outputs.last_hidden_state.squeeze(0)
        input_ids = encoded["input_ids"].squeeze(0)

        target_tokens = self.tokenizer.encode(target_word, add_special_tokens=False)

        ids = input_ids.tolist()
        for i in range(len(ids) - len(target_tokens) + 1):
            if ids[i:i+len(target_tokens)] == target_tokens:
                return token_embeddings[i:i+len(target_tokens)].mean(dim=0).cpu().numpy()

        return None

    def process_csv(self, csv_path):
        df = pd.read_csv(csv_path)

        embeddings = []
        valid_rows = []

        for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Embedding {Path(csv_path).name}"):

            emb = self.get_word_embedding(row["sentence"], row["word"])

            if emb is not None:
                embeddings.append(emb)
                valid_rows.append(idx)

        df_valid = df.iloc[valid_rows].reset_index(drop=True)
        embeddings = np.array(embeddings)

        return df_valid, embeddings

def main():

    CSV_DIR = Path("/content/formal_csvs3")          # your folder
    EMBEDDING_DIR = Path("/content/embeddings_formal3")
    FINAL_DIR = Path("/content/final_embeddings_formal3")

    EMBEDDING_DIR.mkdir(exist_ok=True)
    FINAL_DIR.mkdir(exist_ok=True)

    generator = ContextualEmbeddingGenerator()

    all_dataframes = []

    csv_files = list(CSV_DIR.glob("*.csv"))
    print(f"\nFound {len(csv_files)} CSV files\n")

    for csv_file in csv_files:

        output_meta = EMBEDDING_DIR / f"{csv_file.stem}_metadata.csv"
        output_emb = EMBEDDING_DIR / f"{csv_file.stem}_embeddings.npy"

        # resume protection
        if output_meta.exists() and output_emb.exists():
            print(f"⏭ Skipping {csv_file.name} (already processed)")
            df_meta = pd.read_csv(output_meta)
            embeddings = np.load(output_emb, allow_pickle=True)
        else:
            print(f"🔹 Processing {csv_file.name}")
            df_meta, embeddings = generator.process_csv(csv_file)

            df_meta.to_csv(output_meta, index=False)
            np.save(output_emb, embeddings)

        # attach embeddings
        df_meta["embedding"] = [emb.tolist() for emb in embeddings]
        all_dataframes.append(df_meta)

    # merge all
    final_df = pd.concat(all_dataframes, ignore_index=True)

    final_path = FINAL_DIR / "ALL_FORMAL_WORDS_WITH_EMBEDDINGS3.csv"
    final_df.to_csv(final_path, index=False)

    print("\n✅ ALL DONE")
    print(f"Total rows: {len(final_df)}")
    print(f"Saved to: {final_path}")


if __name__ == "__main__":
    main()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Found 15 CSV files

🔹 Processing follower.csv


Embedding follower.csv: 100%|██████████| 3171/3171 [01:54<00:00, 27.75it/s]


🔹 Processing farm.csv


...

🔹 Processing evergreen.csv


Embedding evergreen.csv: 100%|██████████| 407/407 [00:13<00:00, 29.22it/s]


🔹 Processing omega.csv


Embedding omega.csv: 100%|██████████| 3373/3373 [02:02<00:00, 27.59it/s]



✅ ALL DONE
Total rows: 167619
Saved to: /content/final_embeddings_formal3/ALL_FORMAL_WORDS_WITH_EMBEDDINGS3.csv
